Requires zadik driver - libusbK (v3.x.x)

NEED TO TURN PUMP OFF AND ON AGAIN BEFORE TRYING AGAIN

In [1]:
import usb.core
import usb.util
import time

class Gilson4020USB:
    def __init__(self, vid=0x1FD0, pid=0x0404):
        self.dev = usb.core.find(idVendor=vid, idProduct=pid)
        if self.dev is None:
            raise ValueError("Gilson 4020 USB device not found.")

        self.dev.set_configuration()
        cfg = self.dev.get_active_configuration()
        self.interface_number = cfg[(0, 0)].bInterfaceNumber

        usb.util.claim_interface(self.dev, self.interface_number)
        print(f"Connected to Gilson 4020 on interface {self.interface_number}")

    def send_home_command(self):
        # Emulate the vendor-specific control transfer for 'Home'
        bmRequestType = 0x40  # Host to device, vendor, device
        bRequest = 3          # Vendor-defined command (matches Wireshark)
        wValue = 0xC068       # Matches value seen during HOME click
        wIndex = 0x0200       # Often used to specify report or channel

        print("[DEBUG] Sending HOME control transfer...")
        self.dev.ctrl_transfer(
            bmRequestType=bmRequestType,
            bRequest=bRequest,
            wValue=wValue,
            wIndex=wIndex,
            data_or_wLength=[]
        )
        print("[DEBUG] HOME command sent.")

    def release(self):
        usb.util.release_interface(self.dev, self.interface_number)
        usb.util.dispose_resources(self.dev)
        print("Released USB interface.")


In [2]:
pump = Gilson4020USB()
pump.send_home_command()
pump.release()


USBError: [Errno 13] Access denied (insufficient permissions)